# KNN — Vizualizáció

**Felelős:** Kasza Dominik Árpád

Ez a notebook **csak vizualizál**. A `06_knn.ipynb` által Drive-ra mentett artefaktokat olvassa be:
- `knn_tuned.joblib`
- `knn_predictions_test.npy`
- `knn_validation_curve_n_neighbors.npz`
- `knn_learning_curve.npz`

+ a `data_io.load_v1_data()`-ból a `y_test`, `X_test`, `feature_names`.

Sklearn fit-mentes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/06_knn_visualize.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [10]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
From https://github.com/Slityak/gepitan-seoul-bike-trip
 * branch            main       -> FETCH_HEAD
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Slityak/gepitan-seoul-bike-trip
 * branch            main       -> FETCH_HEAD
Already up to date.
Working dir: /content/gepitan-beadando
In Colab: True
Branch: main


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.config import MODELS_DIR, FIGURES_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.visualization import (
    plot_error_by_feature_quantile,
    plot_learning_curve,
    plot_predicted_vs_actual,
    plot_residuals,
    plot_validation_curve,
)

ensure_dirs()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Adatok és mentett artefaktok betöltése

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

best_pipeline = joblib.load(MODELS_DIR / 'knn_tuned.joblib')
y_pred_tuned  = np.load(MODELS_DIR / 'knn_predictions_test.npy')
y_test_knn    = np.load(MODELS_DIR / 'knn_y_test_sample.npy')
test_idx      = np.load(MODELS_DIR / 'knn_test_idx.npy')
X_test_knn    = X_test[test_idx]
vc = np.load(MODELS_DIR / 'knn_validation_curve_n_neighbors.npz', allow_pickle=True)
lc = np.load(MODELS_DIR / 'knn_learning_curve.npz')

knn_step = best_pipeline.named_steps['knn']
print(f'KNN paraméterek: n_neighbors={knn_step.n_neighbors}, weights={knn_step.weights}, metric={knn_step.metric}')
print(f'y_pred shape: {y_pred_tuned.shape}, y_test_knn shape: {y_test_knn.shape}')

## 3. Predicted vs actual

In [ ]:
plot_predicted_vs_actual(
    y_test_knn, y_pred_tuned,
    model_name='KNN (tuned)',
    save_path='knn_predicted_vs_actual.png',
)
plt.show()

## 4. Residuals

In [ ]:
plot_residuals(
    y_test_knn, y_pred_tuned,
    model_name='KNN (tuned)',
    save_path='knn_residuals.png',
)
plt.show()

## 5. Validation curve (n_neighbors)

In [ ]:
best_weights = str(vc['best_weights'])
best_metric  = str(vc['best_metric'])

plot_validation_curve(
    param_range_labels=list(vc['param_range_labels']),
    train_mae=vc['train_mae'],
    test_mae=vc['test_mae'],
    param_name='n_neighbors',
    model_name=f'KNN (weights={best_weights}, metric={best_metric})',
    save_path='knn_validation_curve_n_neighbors.png',
)
plt.show()

## 6. Learning curve

In [ ]:
plot_learning_curve(
    sizes=lc['sizes'],
    train_mae=lc['train_mae'],
    test_mae=lc['test_mae'],
    model_name='KNN (tuned)',
    save_path='knn_learning_curve.png',
)
plt.show()

## 7. Hiba feature szerint

In [ ]:
hav_idx = feature_names.index('Haversine')
plot_error_by_feature_quantile(
    y_test_knn, y_pred_tuned, X_test_knn[:, hav_idx],
    feature_name='Haversine', n_bins=10,
    model_name='KNN (tuned)',
    save_path='knn_error_by_haversine.png',
)
plt.show()

phour_idx = feature_names.index('Phour')
plot_error_by_feature_quantile(
    y_test_knn, y_pred_tuned, X_test_knn[:, phour_idx],
    feature_name='Phour', n_bins=10,
    model_name='KNN (tuned)',
    save_path='knn_error_by_phour.png',
)
plt.show()